# BTS Digital Twin - M0 Mask (`hcm0031`)

Notebook này nằm hoàn toàn trong `trick/` và chạy workflow `M0-mask` để đo metric chỉ trên pixel tower.

Mặc định notebook sẽ:
- clone repo theo branch bạn vừa `commit/push`
- cài dependency đủ để tạo `bootstrap mask` và chấm `masked metric`
- dùng render có sẵn trong repo nếu có, hoặc render lại nếu bạn bật `RUN_RENDER = True`
- zip toàn bộ output trong `trick/hcm0031/m0_mask/` để gửi lại


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/ThongLuc2k3/BTS-Digital-Twin.git'
REPO_BRANCH = 'coordination/round1-status'
GITHUB_TOKEN = ''

SCENE = 'hcm0031'
WORKDIR = '/content' if Path('/content').exists() else '/kaggle/working'
PROJECT_DIR = f'{WORKDIR}/BTS-Digital-Twin'

RUN_RENDER = False
MASK_MODE = 'bootstrap'  # 'bootstrap' or 'manual'

GS_REPO_URL = 'https://github.com/graphdeco-inria/gaussian-splatting.git'
GS_REPO_REF = 'main'
GS_REPO_DIR = f'{WORKDIR}/gaussian-splatting'

DATASET_ROOT = f'{PROJECT_DIR}/Dataset/VAI_NVS_DATA/phase1/public_set'
TRICK_DIR = f'{PROJECT_DIR}/trick'
BOOTSTRAP_DIR = f'{TRICK_DIR}/hcm0031/m0_mask/bootstrap_masks'
MANUAL_DIR = f'{TRICK_DIR}/hcm0031/m0_mask/manual_masks'
METRICS_DIR = f'{TRICK_DIR}/hcm0031/m0_mask/metrics'
RENDERS_DIR = f'{PROJECT_DIR}/pipeline/work/{SCENE}/renders'
MODEL_DIR = f'{PROJECT_DIR}/pipeline/work/{SCENE}/gs_model'
RESULT_ZIP = f'{WORKDIR}/m0_mask_{SCENE}_artifacts.zip'

print('WORKDIR =', WORKDIR)
print('PROJECT_DIR =', PROJECT_DIR)
print('TRICK_DIR =', TRICK_DIR)
print('RUN_RENDER =', RUN_RENDER)
print('MASK_MODE =', MASK_MODE)


In [ ]:
import shutil
import subprocess
from pathlib import Path

clone_url = REPO_URL
if GITHUB_TOKEN and 'github.com' in REPO_URL:
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

if Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, clone_url, PROJECT_DIR], check=True)
print('Cloned repo to', PROJECT_DIR)


In [ ]:
import subprocess

subprocess.run([
    'bash', '-lc',
    'python3 -m pip install --upgrade pip && '
    'python3 -m pip install pillow numpy lpips scikit-image torch torchvision'
], check=True)
print('Dependencies installed')


## Optional render

Mặc định notebook sẽ dùng render sẵn có trong repo. Nếu thiếu render, đổi `RUN_RENDER = True` ở cell config rồi chạy lại từ đầu.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if RUN_RENDER:
    if Path(GS_REPO_DIR).exists():
        shutil.rmtree(GS_REPO_DIR)
    subprocess.run(['git', 'clone', '--recursive', '--branch', GS_REPO_REF, GS_REPO_URL, GS_REPO_DIR], check=True)
    env = os.environ.copy()
    env['GS_REPO'] = GS_REPO_DIR
    subprocess.run([
        'python3', f'{PROJECT_DIR}/pipeline/scripts/render_round1_test_poses.py',
        '--scene', SCENE,
        '--dataset_root', DATASET_ROOT,
        '--model_dir', MODEL_DIR,
        '--iteration', '-1',
        '--out_dir', RENDERS_DIR,
    ], check=True, env=env)
    print('Rendered test poses to', RENDERS_DIR)
else:
    print('Skip render. Expect renders at', RENDERS_DIR)


In [ ]:
import subprocess

subprocess.run([
    'python3', f'{TRICK_DIR}/scripts/bootstrap_tower_masks.py',
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--tower_bbox3d_json', f'{PROJECT_DIR}/pipeline/work/{SCENE}/tower_bbox3d.json',
    '--out_dir', BOOTSTRAP_DIR,
    '--dilate_px', '12',
], check=True)
print('Bootstrap masks saved to', BOOTSTRAP_DIR)


## Manual mask note

Nếu chưa sửa tay mask, giữ `MASK_MODE = 'bootstrap'` để lấy kết quả nhanh. Nếu đã sửa tay và upload lại vào `manual_masks/`, đổi `MASK_MODE = 'manual'` trước khi chạy cell metric.


In [ ]:
import subprocess
from pathlib import Path

mask_dir = MANUAL_DIR if MASK_MODE == 'manual' else BOOTSTRAP_DIR
if MASK_MODE == 'manual' and not any(Path(MANUAL_DIR).glob('*.png')):
    raise RuntimeError(f'MASK_MODE=manual but no PNG masks found in {MANUAL_DIR}')

subprocess.run([
    'python3', f'{TRICK_DIR}/scripts/eval_round1_mask_metrics.py',
    '--scene', SCENE,
    '--dataset_root', DATASET_ROOT,
    '--renders_dir', RENDERS_DIR,
    '--mask_dir', mask_dir,
    '--out_csv', f'{METRICS_DIR}/masked_eval.csv',
    '--summary_txt', f'{METRICS_DIR}/masked_eval_summary.txt',
    '--psnr_max', '50.0',
    '--min_coverage', '0.001',
], check=True)

print(Path(f'{METRICS_DIR}/masked_eval_summary.txt').read_text(encoding='utf-8'))


In [ ]:
import shutil
from pathlib import Path

artifact_root = Path(f'{TRICK_DIR}/hcm0031/m0_mask')
zip_base = RESULT_ZIP.removesuffix('.zip') if RESULT_ZIP.endswith('.zip') else RESULT_ZIP
shutil.make_archive(zip_base, 'zip', artifact_root)
print('Saved zip:', RESULT_ZIP)

for p in sorted(artifact_root.rglob('*')):
    if p.is_file() and p.suffix in {'.txt', '.csv'}:
        print(p.relative_to(artifact_root))


## Gửi lại cho mình

Chỉ cần gửi lại 2 thứ:
- `trick/hcm0031/m0_mask/metrics/masked_eval_summary.txt`
- `m0_mask_hcm0031_artifacts.zip`
